In [ ]:
%matplotlib widget
import warnings
import inspect
import matplotlib.pyplot as plt
import IPython.display
from cued_sf2_lab.familiarisation import load_mat_img, plot_image
import numpy as np
from typing import Tuple

<figure id="figure-5">
<div style="background-color: white">

![](figures/dwt.svg)

</div>

<figcaption style="text-align: center">

Figure 5: An $L$ level binary discrete wavelet transform.</figcaption></figure>


# 9 The Discrete Wavelet Transform (DWT)

The final method of energy compaction that we shall investigate, is the
discrete wavelet transform. In some ways this attempts to combine the best features of
the Laplacian pyramid and the DCT:

- Like the pyramid, the DWT analyses the image at a range of different
  scales (levels) and employs symmetrical filters;

- Like the DCT, the DWT avoids any expansion in the number of coefficients.

Wavelet theory was evolved by mathematicians during the 1980's. As with the LBT, we shall not attempt to teach this theory here, just illustrate a relatively simple form of it.

Wavelets are short waveforms which are usually the impulse responses of
filters. Wavelet transforms employ banks of bandpass filters, whose impulse
responses are scaled versions of each other, in
order to get pass-bands in different parts of the frequency spectrum. If the
impulse response of a filter is scaled in time by a factor $a$, then the
filter frequency response is scaled by the factor $1/a$. Typically $a = 2$
from one filter to the next, and each bandpass filter is designed to pass a
2:1 range of frequencies (one octave). We can split an image up using wavelets by a process known as a _binary wavelet tree_.


## 9.1 The binary wavelet tree

We start in 1-D with the
simplest possible pair of filters, operating on just two input samples, $x_n$
and $x_{n-1}$. The two filter outputs, $u_n$ and $v_n$ at time $n$ are
given by:

$$
 u_n = \tfrac{1}{2} (x_n + x_{n-1}) \quad \text{and} \quad
 v_n = \tfrac{1}{2} (x_n - x_{n-1})
$$

The first filter averages adjacent samples, and so rejects the higher
frequency components of $x$, while the second filter differences these
samples, and so rejects the lower frequency components. These filters are
known as the _analysis_ filter pair, $H_1(z) = \tfrac{1}{2} (1 + z^{-1})$
and $H_2(z) = \tfrac{1}{2} (1 - z^{-1})$. It is clear that we can recover the
two input samples from the filter outputs using:

$$
 x_n = u_n + v_n \quad \text{and} \quad x_{n-1} = u_n - v_n
$$

Next it is important to note that we need only retain the samples of $u_n$
and $v_n$ at even values of $n$ in order to be able to recover all the
original samples of $x$. Hence $u$ and $v$ may be decimated 2:1 and still
allow perfect reconstruction of $x$. If $x$ is a finite length vector (e.g. a
row of image pixels), then $u$ and $v$ are each half as long as $x$, so the
total number of samples is preserved by the transformation.

A wavelet binary tree may be constructed using these filters, by using an
identical pair, $H_1$ and $H_2$, to filter the decimated lowpass signal
$u_{2n}$, to give a pair of outputs, $uu_{2n}$ and $uv_{2n}$, representing
the lower and upper halves of the first low band. These may again be
decimated 2:1 and still permit perfect reconstruction of $u$. This process
may be continued as often as desired: each time splitting the lowest band in
two, and decimating the sample rate of the filter outputs by 2:1. At each
stage the bandwidth of the two lowest filters is halved, and their impulse
responses are doubled in length. The total number of output samples remains
constant, however many stages are used.

For example, if $f_s$ is the input sample rate, a 3-stage binary tree will
split the input signal bandwidth of 0 to $f_s/2$ into the following four
bands:

$$
0 \rightarrow f_s/16; \ \ f_s/16 \rightarrow f_s/8; \ \ f_s/8 \rightarrow
f_s/4;  \ \ f_s/4 \rightarrow f_s/2.
$$

The very simple filters, given above, do not generate a filter tree with
good characteristics, since the wavelets turn out to be just a pair of
square pulses. These generate _blocking_ artefacts when used for image
compression (in fact they are equivalent to the 2 point ($N=2$)
DCT). A better set of filters are the LeGall 5 and 3 tap pair,
given by:

$$
 u_n = \tfrac{1}{8} (-x_{n+2} + 2 x_{n+1} + 6 x_n + 2 x_{n-1} - x_{n-2})
  \quad \text{ and }  \quad
 v_{n+1} = \tfrac{1}{4} (-x_{n+2} + 2 x_{n+1} - x_n)
$$

If $u$ and $v$ are decimated by 2 by choosing even $n$ only, the lowband outputs
$u_n$ are centred on the even samples, and the highband outputs $v_{n+1}$ are
centred on the odd samples. This is very important to allow perfect
reconstruction of $x$ from $u$ and $v$.

The equations for reconstruction may be obtained by solving the above to get:

$$
\begin{aligned}
x_n &= \tfrac{1}{2} (-v_{n+1} + 2 u_n - v_{n-1}) \\
x_{n+1} &= \tfrac{1}{2} (x_{n+2} + 4 v_{n+1} + x_n) = \tfrac{1}{4} (-v_{n+3} + 2 u_{n+2} + 6 v_{n+1} + 2 u_n - v_{n-1})
\end{aligned}
$$

In general, most analysis filters will not yield such simple reconstruction
solutions, and the design of suitable filters is a non-trivial topic that we
shall not cover here.


## 9.2 Applying the DWT to images

As with the DCT, the 2-D DWT may be obtained by applying a 1-D transform to
first the rows and then the columns of an image.

Start by loading the Lighthouse image and defining the two LeGall
filters given above:


In [ ]:
X, _ = load_mat_img(img='lighthouse.mat', img_info='X', cmap_info={'map', 'map2'})
X = X - 128.0
h1 = np.array([-1, 2, 6, 2, -1])/8
h2 = np.array([-1, 2, -1])/4

We can use the function `rowdec` from the pyramid work, to
produce a decimated and lowpass filtered version of the rows of
`X` (remembering to subtract 128 as before) using:


In [ ]:
from cued_sf2_lab.laplacian_pyramid import rowdec
U = rowdec(X, h1)

To get the high-pass image `V`, it is important to align the decimated
samples with the odd columns of `X` (assuming the first column is $n = 0$)
whereas `U` is aligned with the even columns. To do this we use a
slightly modified version of `rowdec`, called `rowdec2`.


In [ ]:
from cued_sf2_lab.laplacian_pyramid import rowdec2
V = rowdec2(X, h2)

<div class="alert alert-block alert-danger">

Display `U` and `V` to see the outputs of the first filter pair
and comment on their relative energies (or standard deviations). Note that `U` and `V` are half the width of `X`, but that `U` is otherwise similar to `X`.</div>


In [ ]:
Eu, Ev = np.sum(U**2), np.sum(V**2)

fig, axs = plt.subplots(1, 2, figsize=(10, 4))
plot_image(U, ax=axs[0]); axs[0].set(title=f'U (LP rows), std={np.std(U):.1f}, E={Eu:.2e}')
plot_image(V, ax=axs[1]); axs[1].set(title=f'V (HP rows), std={np.std(V):.1f}, E={Ev:.2e}')
fig.tight_layout()
fig.savefig('zach/images/dwt_uv_display.png', dpi=150, bbox_inches='tight')

print(f'X:  std={np.std(X):.1f}, energy={np.sum(X**2):.0f}')
print(f'U:  std={np.std(U):.1f}, energy={Eu:.0f}')
print(f'V:  std={np.std(V):.1f}, energy={Ev:.0f}')
print(f'U+V energy: {Eu+Ev:.0f}')
print(f'V/U energy ratio: {Ev/Eu*100:.1f}%')

U is half the width of X but otherwise looks similar — most energy is in the lowpass. V is mostly near zero, with only edges visible; its standard deviation is much lower, confirming the energy compaction of the LeGall 5-tap lowpass.


Now filter the columns of `U` and `V` using `rowdec / rowdec2` with the transpose operator:


In [ ]:
UU = rowdec(U.T, h1).T
UV = rowdec2(U.T, h2).T
VU = rowdec(V.T, h1).T
VV = rowdec2(V.T, h2).T

<div class="alert alert-block alert-danger">

Display `np.block([[UU, VU], [UV, VV]])`, and comment
on what sort of edges or features are selected by each filter. You may need to multiply the high-pass images by a factor $k > 1$ to display them clearly. Why is this?</div>


In [ ]:
k = 3
fig, axs = plt.subplots(2, 2, figsize=(8, 8))

for ax, img, label in zip(axs.ravel(), [UU, k*VU, k*UV, k*VV],
    ['UU (LP rows, LP cols)', f'VU ×{k} (HP rows, LP cols)\nvertical edges',
     f'UV ×{k} (LP rows, HP cols)\nhorizontal edges', f'VV ×{k} (HP rows, HP cols)\ndiagonal features']):
    plot_image(img, ax=ax)
    ax.set(title=label, xticks=[], yticks=[])

fig.suptitle(f'Single-level 2-D DWT (HP sub-images scaled by {k}×)', fontsize=12)
fig.tight_layout()
fig.savefig('zach/images/dwt_composite.png', dpi=150, bbox_inches='tight')

**UU** (top-left): LP in both dimensions — a quarter-size replica of X. **VU** (top-right): HP rows, LP columns — picks out vertical edges. **UV** (bottom-left): LP rows, HP columns — picks out horizontal edges. **VV** (bottom-right): HP both — diagonal/corner features. The HP images need a display gain factor $k>1$ because their values cluster tightly near zero (energy compaction).


We must now check that it is possible to recover the image from
these sub-images, using reconstruction filters, `g1` and `g2`, and the functions, `rowint` and `rowint2` (which
is modified in a similar way to `rowdec2` to allow correct
alignment of the high-pass samples). To reconstruct `Ur` and
`Vr` from `UU`, `UV`, `VU` and `VV` use:


In [ ]:
from cued_sf2_lab.laplacian_pyramid import rowint, rowint2

g1 = np.array([1, 2, 1])/2
g2 = np.array([-1, -2, 6, -2, -1])/4
Ur = rowint(UU.T, g1).T + rowint2(UV.T, g2).T
Vr = rowint(VU.T, g1).T + rowint2(VV.T, g2).T

Note the gain of 2 in the reconstruction filters, `g1` and
`g2` (to compensate for losing half the samples in the
decimation / interpolation processes). These filters are also
not quite the same as those that might be inferred from the
equations for $x_n$ and $x_{n+1}$ on the previous page. This is
because `g1` defines how _only_ the $u$ samples contribute
both to the even and odd samples of $x$, while `g2` defines
how the $v$ samples contribute.

Check that `Ur` and `Vr` are the same as `U` and
`V`, and then reconstruct `Xr` from these:


In [ ]:
np.testing.assert_equal(Ur, U)
np.testing.assert_equal(Vr, V)
print('Ur == U and Vr == V: OK')

Xr = rowint(Ur, g1) + rowint2(Vr, g2)
print(f'Max |Xr - X|: {np.max(np.abs(Xr - X)):.2e}')

The above operations are a bit tedious to repeat if we want to
apply the DWT recursively to obtain several levels of filtering,
so we have written a pair of functions, `dwt` and `idwt`, to perform the 2-D analysis and reconstruction
operations. Examine these to see that they perform the same
operations as above, except that the transformed sub-images are
stored as parts of a single matrix, the same size as `X`,
rather than as separate matrices.


In [ ]:
from cued_sf2_lab.dwt import dwt, idwt

In [ ]:
Y = dwt(X)
Xr = idwt(Y)

fig, axs = plt.subplots(1, 2)
plot_image(Y, ax=axs[0]); axs[0].set(title='Y = dwt(X)')
plot_image(Xr, ax=axs[1]); axs[1].set(title='Xr = idwt(Y)')
print(f'Max |Xr - X|: {np.max(np.abs(Xr - X)):.2e}')

### Multi-level DWT

Implement a multilevel DWT by iteratively applying `dwt` to the top-left sub-image of `Y`.


In [ ]:
m = 256; Y = dwt(X)
fig, axs = plt.subplots(1, 4, figsize=(16, 4))
plot_image(Y, ax=axs[0]); axs[0].set(title='1 level')
for i, ax in enumerate(axs[1:], 2):
    m //= 2
    Y[:m, :m] = dwt(Y[:m, :m])
    plot_image(Y, ax=ax); ax.set(title=f'{i} levels')
fig.tight_layout()
fig.savefig('zach/images/dwt_multilevel.png', dpi=150, bbox_inches='tight')

Reconstruct the image from the 4-level DWT by walking back from the smallest block:


In [ ]:
# Y currently has 4 levels from above
n_levels = 4
m = Y.shape[0] // (2**(n_levels - 1))  # = 32
for _ in range(n_levels):
    Y[:m, :m] = idwt(Y[:m, :m])
    m *= 2
print(f'Max |X - reconstructed|: {np.max(np.abs(X - Y)):.2e}')

## 9.3 Quantisation and coding efficiency

First rewrite the sequences of operations required to perform
$n$ levels of DWT and inverse DWT as two separate functions, `nlevdwt` and `nlevidwt`. `nlevdwt` should transform
`X` into `Y`, and `nlevidwt` should inverse
transform a quantised set of sub-images `Yq` into the
reconstructed image `Z`. Check your functions by ensuring
that `Z` is the same as `X` if `Yq = Y`.


In [ ]:
def nlevdwt(X, n):
    """n-level 2D DWT. Returns Y same size as X with sub-images packed in-place."""
    Y = X.copy().astype(float)
    m = Y.shape[0]
    for _ in range(n):
        Y[:m, :m] = dwt(Y[:m, :m])
        m //= 2
    return Y

def nlevidwt(Y, n):
    """Inverse of nlevdwt. Reconstructs X from n-level DWT coefficients."""
    Z = Y.copy().astype(float)
    m = Z.shape[0] // (2**(n - 1))
    for _ in range(n):
        Z[:m, :m] = idwt(Z[:m, :m])
        m *= 2
    return Z

In [ ]:
for n in range(1, 6):
    Y = nlevdwt(X, n)
    Z = nlevidwt(Y, n)
    print(f'n={n}: max |X - Z| = {np.max(np.abs(X - Z)):.2e}')

### `quantdwt`

Quantise the sub-images of `Y` according to a $3 \times (n+1)$ matrix `dwtstep[k,i]` of
step-sizes, where $k \in \{0,1,2\}$ corresponds to each of the three high-pass images at level $i$ (top-right, bottom-left, bottom-right), and the final low-pass image is quantised with `dwtstep[0,n]`.


In [ ]:
from cued_sf2_lab.laplacian_pyramid import bpp, quantise

def quantdwt(Y: np.ndarray, dwtstep: np.ndarray) -> Tuple[np.ndarray, np.ndarray]:
    """
    Quantise DWT sub-images with per-band step sizes.

    Parameters:
        Y: output of nlevdwt(X, n)
        dwtstep: array of shape (3, n+1)
    Returns:
        Yq: quantised Y
        dwtent: array of shape (3, n+1) with entropies (bpp) per sub-image
    """
    n = dwtstep.shape[1] - 1
    m = Y.shape[0]
    Yq = Y.copy()
    dwtent = np.zeros_like(dwtstep)

    for i in range(n):
        m_i = m // (2**i)
        half = m_i // 2
        # (k=0) top-right, (k=1) bottom-left, (k=2) bottom-right
        slices = [
            (slice(0, half), slice(half, m_i)),
            (slice(half, m_i), slice(0, half)),
            (slice(half, m_i), slice(half, m_i)),
        ]
        for k, (rs, cs) in enumerate(slices):
            Yq[rs, cs] = quantise(Yq[rs, cs], dwtstep[k, i])
            dwtent[k, i] = bpp(Yq[rs, cs])

    # final LP
    m_n = m // (2**n)
    Yq[:m_n, :m_n] = quantise(Yq[:m_n, :m_n], dwtstep[0, n])
    dwtent[0, n] = bpp(Yq[:m_n, :m_n])

    return Yq, dwtent


def dwt_total_bits(dwtent, m, n):
    """Total bits from the entropy matrix returned by quantdwt."""
    total = 0.0
    for i in range(n):
        npix = (m // 2**(i+1))**2
        total += sum(dwtent[k, i] * npix for k in range(3))
    m_n = m // (2**n)
    total += dwtent[0, n] * m_n**2
    return total

### Helpers and quick constant-step test


In [ ]:
def dwt_quantize_const(Xi, step, n):
    """Constant-step DWT quantisation. Returns (Z, rms, bits)."""
    Y = nlevdwt(Xi, n)
    dwtstep = np.full((3, n + 1), step)
    Yq, dwtent = quantdwt(Y, dwtstep)
    Z = nlevidwt(Yq, n)
    return Z, np.std(Xi - Z), dwt_total_bits(dwtent, Xi.shape[0], n)

def dwt_quantize_layered(Xi, dwtstep, n):
    """Per-band step DWT quantisation. Returns (Z, rms, bits)."""
    Y = nlevdwt(Xi, n)
    Yq, dwtent = quantdwt(Y, dwtstep)
    Z = nlevidwt(Yq, n)
    return Z, np.std(Xi - Z), dwt_total_bits(dwtent, Xi.shape[0], n)

def dwt_impulse_ratios(m, n):
    """Step-size ratios for equal-MSE, normalised so top-right level-0 = 1."""
    energies = np.zeros((3, n + 1))
    slices_for_level = []
    for i in range(n):
        m_i = m // (2**i)
        half = m_i // 2
        slices_for_level.append([
            (slice(0, half), slice(half, m_i)),
            (slice(half, m_i), slice(0, half)),
            (slice(half, m_i), slice(half, m_i)),
        ])
    for i in range(n):
        for k in range(3):
            Y = np.zeros((m, m))
            rs, cs = slices_for_level[i][k]
            Y[(rs.start + rs.stop) // 2, (cs.start + cs.stop) // 2] = 1
            Z = nlevidwt(Y, n)
            energies[k, i] = np.sum(Z**2)
    # final LP
    m_n = m // (2**n)
    Y = np.zeros((m, m))
    Y[m_n // 2, m_n // 2] = 1
    Z = nlevidwt(Y, n)
    energies[0, n] = np.sum(Z**2)
    # unused entries: set to inf so ratio → 0 without warnings
    energies[1, n] = np.inf
    energies[2, n] = np.inf
    # step proportional to 1/sqrt(energy), normalise to energies[0, 0]
    ratios = np.sqrt(energies[0, 0] / energies)
    return ratios

In [ ]:
step = 17
rms_direct = np.std(X - quantise(X, step))
bits_direct = bpp(quantise(X, step)) * X.size
print(f'Direct quantisation: RMS={rms_direct:.3f}, bits={bits_direct:.0f}\n')

for n in range(1, 6):
    Z, rms, bits = dwt_quantize_const(X, step, n)
    print(f'n={n}: bits={bits:.0f}, CR={bits_direct/bits:.3f}, RMS={rms:.3f}')

All of our experiments thus far have been performed on only one image. At this stage it is worth starting to experiment with the additional `Bridge` image, as well as Lighthouse. Bridge contains a lot more fine detail and may not lead to the same conclusions regarding performance.


In [ ]:
Xb, _ = load_mat_img(img='bridge.mat', img_info='X', cmap_info={'map'})
Xb = Xb - 128.0

fig, ax = plt.subplots()
plot_image(Xb, ax=ax)
ax.set(title='bridge.mat')

<div class="alert alert-block alert-danger">

Investigate the performance of
both an equal-step-size and an equal-MSE scheme (follow a similar procedure as you used for the Laplacian Pyramid to find the appropriate step-size ratios). Hence determine how many levels of DWT are reasonably optimal for the Lighthouse and Bridge images. Also evaluate the subjective quality of your reconstructed images, and comment on how this depends on $n$ and on the way that step-sizes are assigned
to the different levels. Once again, for each image choose quantisation steps such that you match the rms error to that for direct quantisation with a step-size of 17.</div>


In [ ]:
step_grid = np.arange(1, 50, 0.1)
scale_grid = np.arange(0.1, 30, 0.1)
ns = [1, 2, 3, 4, 5, 6]

for img_name, Xi in [('Lighthouse', X), ('Bridge', Xb)]:
    rms_ref = np.std(Xi - quantise(Xi, 17))
    bits_ref = bpp(quantise(Xi, 17)) * Xi.size
    m = Xi.shape[0]

    print(f'\n{"="*60}')
    print(f'{img_name}: rms_ref={rms_ref:.3f}, bits_ref={bits_ref:.0f}')
    print(f'{"="*60}')

    cr_const, cr_eqmse = {}, {}
    step_const, step_eqmse = {}, {}

    for n in ns:
        # --- constant step ---
        errs = np.array([dwt_quantize_const(Xi, s, n)[1] for s in step_grid])
        best = step_grid[np.argmin(np.abs(errs - rms_ref))]
        _, rms, bits = dwt_quantize_const(Xi, best, n)
        cr_const[n] = bits_ref / bits
        step_const[n] = best
        print(f'n={n} constant:  step={best:.1f}, RMS={rms:.3f}, CR={cr_const[n]:.3f}')

        # --- equal-MSE ---
        ratios = dwt_impulse_ratios(m, n)
        errs_eq = []
        for sc in scale_grid:
            dwtstep = sc * ratios
            dwtstep[1, n] = dwtstep[0, n]
            dwtstep[2, n] = dwtstep[0, n]
            _, rms_s, _ = dwt_quantize_layered(Xi, dwtstep, n)
            errs_eq.append(rms_s)
        errs_eq = np.array(errs_eq)
        best_sc = scale_grid[np.argmin(np.abs(errs_eq - rms_ref))]
        dwtstep = best_sc * ratios
        dwtstep[1, n] = dwtstep[0, n]
        dwtstep[2, n] = dwtstep[0, n]
        _, rms, bits = dwt_quantize_layered(Xi, dwtstep, n)
        cr_eqmse[n] = bits_ref / bits
        step_eqmse[n] = best_sc
        print(f'n={n} equal-MSE: scale={best_sc:.1f}, RMS={rms:.3f}, CR={cr_eqmse[n]:.3f}')

    # --- two subplots: step size (left), CR (right) ---
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

    ax1.plot(ns, [step_const[n] for n in ns], 'o-', label='constant step size $\\Delta$')
    ax1.plot(ns, [step_eqmse[n] for n in ns], 's-', label='equal-MSE scale $s$')
    ax1.set(xlabel='DWT levels $n$', ylabel='matched step size / scale', xticks=ns)
    ax1.legend(); ax1.grid(True)
    ax1.set_title('Step size / scale vs depth')

    ax2.plot(ns, [cr_const[n] for n in ns], 'o-', label='constant step')
    ax2.plot(ns, [cr_eqmse[n] for n in ns], 's-', label='equal-MSE')
    ax2.set(xlabel='DWT levels $n$', ylabel='compression ratio', xticks=ns)
    ax2.legend(); ax2.grid(True)
    ax2.set_title('CR vs depth')

    fig.suptitle(f'{img_name}: at matched RMS = {rms_ref:.2f}', fontsize=12)
    fig.tight_layout()
    fig.savefig(f'zach/images/dwt_cr_{img_name.lower()}.png', dpi=150, bbox_inches='tight')

### Understanding the step size plots

**Constant step ($\Delta$):** Every sub-image at every level gets the same quantisation step $\Delta$. We sweep $\Delta$ until the reconstruction RMS matches direct quantisation at step 17. Deeper pyramids accumulate more noise (each level adds noise through the decoder), so to keep RMS fixed we must shrink $\Delta$ — hence the step size falls with depth.

**Equal-MSE scale ($s$):** Each sub-image gets a _different_ step size $\Delta_i = s \cdot r_i$, where $r_i$ is the impulse response ratio for that sub-image (measured once, fixed). The ratios $r_i$ ensure each sub-image contributes equally to the output MSE. We then sweep the single overall scale $s$ until the total RMS matches. Because the ratios already compensate for depth, $s$ stays roughly constant as $n$ increases — the equal-MSE scheme automatically allocates finer steps to deeper levels without needing to shrink the overall scale.


In [ ]:
# Visual comparison: n=3 and n=4, constant-step and equal-MSE, for both images
for img_name, Xi in [('Lighthouse', X), ('Bridge', Xb)]:
    rms_ref = np.std(Xi - quantise(Xi, 17))
    bits_ref = bpp(quantise(Xi, 17)) * Xi.size
    m = Xi.shape[0]

    panels = []
    for n in [3, 4]:
        # constant step
        errs = np.array([dwt_quantize_const(Xi, s, n)[1] for s in step_grid])
        best = step_grid[np.argmin(np.abs(errs - rms_ref))]
        Z_c, rms_c, bits_c = dwt_quantize_const(Xi, best, n)
        panels.append((Z_c, f'const n={n}, $\\Delta$={best:.1f}\nCR={bits_ref/bits_c:.2f}, RMS={rms_c:.2f}'))

        # equal-MSE
        ratios = dwt_impulse_ratios(m, n)
        errs_eq = []
        for sc in scale_grid:
            dwtstep = sc * ratios
            dwtstep[1, n] = dwtstep[0, n]; dwtstep[2, n] = dwtstep[0, n]
            _, rms_s, _ = dwt_quantize_layered(Xi, dwtstep, n)
            errs_eq.append(rms_s)
        best_sc = scale_grid[np.argmin(np.abs(np.array(errs_eq) - rms_ref))]
        dwtstep = best_sc * ratios
        dwtstep[1, n] = dwtstep[0, n]; dwtstep[2, n] = dwtstep[0, n]
        Z_e, rms_e, bits_e = dwt_quantize_layered(Xi, dwtstep, n)
        panels.append((Z_e, f'eq-MSE n={n}, s={best_sc:.1f}\nCR={bits_ref/bits_e:.2f}, RMS={rms_e:.2f}'))

    # 1 row x 6 cols: Original, Direct, const n=3, eq-MSE n=3, const n=4, eq-MSE n=4
    fig, axs = plt.subplots(1, 6, figsize=(20, 3.5))
    plot_image(Xi, ax=axs[0]); axs[0].set(title='Original', xticks=[], yticks=[])
    plot_image(quantise(Xi, 17), ax=axs[1]); axs[1].set(title=f'Direct\nRMS={rms_ref:.2f}', xticks=[], yticks=[])
    for ax, (Z, title) in zip(axs[2:], panels):
        plot_image(Z, ax=ax)
        ax.set(title=title, xticks=[], yticks=[])
    fig.suptitle(img_name, fontsize=13)
    fig.tight_layout()
    fig.savefig(f'zach/images/dwt_visual_{img_name.lower()}.png', dpi=150, bbox_inches='tight')

In [ ]:
# Cropped visual comparison: n=3 and n=4, constant-step and equal-MSE, for both images
r0, r1, c0, c1 = 100, 180, 50, 180  # fence + sky crop

for img_name, Xi in [('Lighthouse', X), ('Bridge', Xb)]:
    rms_ref = np.std(Xi - quantise(Xi, 17))
    bits_ref = bpp(quantise(Xi, 17)) * Xi.size
    m = Xi.shape[0]

    panels = []
    for n in [3, 4]:
        # constant step
        errs = np.array([dwt_quantize_const(Xi, s, n)[1] for s in step_grid])
        best = step_grid[np.argmin(np.abs(errs - rms_ref))]
        Z_c, rms_c, bits_c = dwt_quantize_const(Xi, best, n)
        panels.append((Z_c, f'const n={n}\nCR={bits_ref/bits_c:.2f}'))

        # equal-MSE
        ratios = dwt_impulse_ratios(m, n)
        errs_eq = []
        for sc in scale_grid:
            dwtstep = sc * ratios
            dwtstep[1, n] = dwtstep[0, n]; dwtstep[2, n] = dwtstep[0, n]
            _, rms_s, _ = dwt_quantize_layered(Xi, dwtstep, n)
            errs_eq.append(rms_s)
        best_sc = scale_grid[np.argmin(np.abs(np.array(errs_eq) - rms_ref))]
        dwtstep = best_sc * ratios
        dwtstep[1, n] = dwtstep[0, n]; dwtstep[2, n] = dwtstep[0, n]
        Z_e, rms_e, bits_e = dwt_quantize_layered(Xi, dwtstep, n)
        panels.append((Z_e, f'eq-MSE n={n}\nCR={bits_ref/bits_e:.2f}'))

    fig, axs = plt.subplots(1, 6, figsize=(20, 4))
    plot_image(Xi[r0:r1, c0:c1], ax=axs[0]); axs[0].set(title='Original', xticks=[], yticks=[])
    plot_image(quantise(Xi, 17)[r0:r1, c0:c1], ax=axs[1]); axs[1].set(title='Direct', xticks=[], yticks=[])
    for ax, (Z, title) in zip(axs[2:], panels):
        plot_image(Z[r0:r1, c0:c1], ax=ax)
        ax.set(title=title, xticks=[], yticks=[])
    fig.suptitle(f'{img_name}: zoomed crop at matched RMS', fontsize=13, y=1.02)
    fig.tight_layout()
    fig.savefig(f'zach/images/dwt_crop_{img_name.lower()}.png', dpi=150, bbox_inches='tight')

## 9.4 Second Interim Report

This report should include the new results from the DCT, LBT and DWT energy
compaction methods in a format that will allow them to be compared with each other and contrasted to the
Laplacian pyramid work in your first report. Again try to answer questions
raised in the text, and also include discussion of any topics that have led to
unexpected results or have proved particularly interesting.


In [ ]:
# --- Reuse imports already available; bring in pyramid + DCT + LBT machinery ---
from cued_sf2_lab.laplacian_pyramid import rowdec, rowint, rowdec2, rowint2, quantise, bpp
from cued_sf2_lab.dct import dct_ii, colxfm, regroup
from cued_sf2_lab.lbt import pot_ii

# --- Pyramid helpers (from nb6) ---
def pykenc(X, h, k):
    Ys, Xk = [], X
    for _ in range(k):
        Xk_next = rowdec(rowdec(Xk, h).T, h).T
        Xk_hat  = rowint(rowint(Xk_next, 2*h).T, 2*h).T
        Ys.append(Xk - Xk_hat)
        Xk = Xk_next
    return Ys, Xk

def pykdec(Ys, Xk, h):
    Z = Xk
    for Y in reversed(Ys):
        Z = rowint(rowint(Z, 2*h).T, 2*h).T + Y
    return Z

def pyr_impulse_energy(layer_idx, k, h, shape):
    Ys, Xk = pykenc(np.zeros(shape), h, k)
    target = Ys[layer_idx] if layer_idx < k else Xk
    r, c = target.shape
    target[r//2, c//2] = 1
    Z0 = pykdec(Ys, Xk, h)
    return np.sum(Z0**2)

def pyr_step_ratios(k, h, shape):
    e = np.array([pyr_impulse_energy(i, k, h, shape) for i in range(k+1)])
    return np.sqrt(e[0] / e)

# --- DCT helper ---
def dctbpp(Yr, N):
    total = 0
    sr, sc = Yr.shape[0]//N, Yr.shape[1]//N
    for r in range(N):
        for c in range(N):
            Ys = Yr[r*sr:(r+1)*sr, c*sc:(c+1)*sc]
            total += bpp(Ys) * Ys.size
    return total

# --- LBT helpers (from nb8) ---
def lbt_enc(Xi, C, Pf):
    N = C.shape[0]; t = np.s_[N//2:-N//2]
    Xp = Xi.copy()
    Xp[t,:] = colxfm(Xp[t,:], Pf)
    Xp[:,t] = colxfm(Xp[:,t].T, Pf).T
    return colxfm(colxfm(Xp, C).T, C).T

def lbt_dec(Y, C, Pr):
    N = C.shape[0]; t = np.s_[N//2:-N//2]
    Z = colxfm(colxfm(Y.T, C.T).T, C.T)
    Zp = Z.copy()
    Zp[:,t] = colxfm(Zp[:,t].T, Pr.T).T
    Zp[t,:] = colxfm(Zp[t,:], Pr.T)
    return Zp

## 9.5 Cross-method comparison

### Rate-distortion curves

Sweep step sizes for all four methods and plot CR vs RMS on the same axes. This shows which method dominates across all quality levels, not just one matched-RMS operating point.


In [ ]:
rd_steps = np.arange(5, 120, 1.0)
h_pyr = 0.25 * np.array([1, 2, 1])
C8 = dct_ii(8)
Pf_lbt, Pr_lbt = pot_ii(8, 1.42)

for img_name, Xi in [('Lighthouse', X), ('Bridge', Xb)]:
    rms_direct_i = np.std(Xi - quantise(Xi, 17))
    rd_curves = {}

    # --- Direct quantisation ---
    rd_direct = []
    for s in rd_steps:
        Xq = quantise(Xi, s)
        rd_direct.append((np.std(Xi - Xq), bpp(Xq) * Xi.size))
    rd_curves['Direct'] = rd_direct

    # --- Pyramid (3-tap, equal-MSE, k=3) ---
    ratios_pyr = pyr_step_ratios(3, h_pyr, Xi.shape)
    rd_pyr = []
    for s in rd_steps:
        Ys, Xk = pykenc(Xi, h_pyr, 3)
        steps = s * ratios_pyr
        Ys_q = [quantise(Y, st) for Y, st in zip(Ys, steps[:-1])]
        Xk_q = quantise(Xk, steps[-1])
        Z = pykdec(Ys_q, Xk_q, h_pyr)
        bits = sum(bpp(Yq)*Yq.size for Yq in Ys_q) + bpp(Xk_q)*Xk_q.size
        rd_pyr.append((np.std(Xi - Z), bits))
    rd_curves['Pyramid'] = rd_pyr

    # --- DCT 8x8 ---
    Y_dct = colxfm(colxfm(Xi, C8).T, C8).T
    rd_dct = []
    for s in rd_steps:
        Yq = quantise(Y_dct, s)
        Z = colxfm(colxfm(Yq.T, C8.T).T, C8.T)
        bits = dctbpp(regroup(Yq, 8), 8)
        rd_dct.append((np.std(Xi - Z), bits))
    rd_curves['DCT 8x8'] = rd_dct

    # --- LBT 8x8 (s=1.42) ---
    Y_lbt = lbt_enc(Xi, C8, Pf_lbt)
    rd_lbt = []
    for s in rd_steps:
        Yq = quantise(Y_lbt, s)
        Z = lbt_dec(Yq, C8, Pr_lbt)
        bits = dctbpp(regroup(Yq, 8), 8)
        rd_lbt.append((np.std(Xi - Z), bits))
    rd_curves['LBT 8x8'] = rd_lbt

    # --- DWT (equal-MSE, n=3) ---
    n_dwt = 3
    ratios_dwt = dwt_impulse_ratios(Xi.shape[0], n_dwt)
    rd_dwt = []
    for s in rd_steps:
        dwtstep = s * ratios_dwt
        dwtstep[1, n_dwt] = dwtstep[0, n_dwt]
        dwtstep[2, n_dwt] = dwtstep[0, n_dwt]
        Z, rms, bits = dwt_quantize_layered(Xi, dwtstep, n_dwt)
        rd_dwt.append((rms, bits))
    rd_curves[f'DWT n={n_dwt}'] = rd_dwt

    # --- Gaussian i.i.d. RD bound ---
    # R(D) = (1/2) log2(sigma^2 / D) bits/sample, D = RMS^2
    # Total bits = N^2 * max(0, (1/2) log2(sigma^2 / RMS^2))
    sigma2 = np.var(Xi)
    N2 = Xi.size
    rms_range = np.linspace(0.5, 15, 200)
    rd_bound = N2 * np.maximum(0, 0.5 * np.log2(sigma2 / rms_range**2))

    # --- Plot ---
    fig, ax = plt.subplots(figsize=(8, 5))
    for label, pts in rd_curves.items():
        rms_arr = np.array([p[0] for p in pts])
        bits_arr = np.array([p[1] for p in pts])
        ax.plot(rms_arr, bits_arr / 1000, '.-', ms=4, label=label)

    ax.plot(rms_range, rd_bound / 1000, 'k--', alpha=0.5, label='Gaussian i.i.d. bound')

    ax.set(xlabel='RMS error', ylabel='Total bits (thousands)',
           title=f'Rate-distortion: {img_name}', xlim=(1, 15), ylim=(0, 500))
    ax.legend()
    ax.grid(True)
    fig.tight_layout()
    fig.savefig(f'zach/images/rd_curves_{img_name.lower()}.png', dpi=150, bbox_inches='tight')

### Cross-method zoomed crop

All methods at matched RMS ≈ 4.86 on the same crop region (fence + sky), showing different artefact profiles side by side.


In [ ]:
rms_ref = np.std(X - quantise(X, 17))
step_grid_fine = np.arange(1, 50, 0.1)
r0, r1, c0, c1 = 100, 180, 50, 180  # fence + sky crop

# --- Pyramid (3-tap, equal-MSE, k=3) at matched RMS ---
ratios_pyr = pyr_step_ratios(3, h_pyr, X.shape)
errs_pyr = []
for s in step_grid_fine:
    steps = s * ratios_pyr
    Ys, Xk = pykenc(X, h_pyr, 3)
    Ys_q = [quantise(Y, st) for Y, st in zip(Ys, steps[:-1])]
    Xk_q = quantise(Xk, steps[-1])
    Z = pykdec(Ys_q, Xk_q, h_pyr)
    errs_pyr.append(np.std(X - Z))
best_s = step_grid_fine[np.argmin(np.abs(np.array(errs_pyr) - rms_ref))]
steps = best_s * ratios_pyr
Ys, Xk = pykenc(X, h_pyr, 3)
Ys_q = [quantise(Y, st) for Y, st in zip(Ys, steps[:-1])]
Z_pyr = pykdec(Ys_q, quantise(Xk, steps[-1]), h_pyr)

# --- DCT 8x8 at matched RMS ---
Y_dct = colxfm(colxfm(X, C8).T, C8).T
errs_dct = [np.std(X - colxfm(colxfm(quantise(Y_dct, s).T, C8.T).T, C8.T)) for s in step_grid_fine]
best_s = step_grid_fine[np.argmin(np.abs(np.array(errs_dct) - rms_ref))]
Z_dct = colxfm(colxfm(quantise(Y_dct, best_s).T, C8.T).T, C8.T)

# --- LBT 8x8 (s=1.42) at matched RMS ---
Y_lbt = lbt_enc(X, C8, Pf_lbt)
errs_lbt = [np.std(X - lbt_dec(quantise(Y_lbt, s), C8, Pr_lbt)) for s in step_grid_fine]
best_s = step_grid_fine[np.argmin(np.abs(np.array(errs_lbt) - rms_ref))]
Z_lbt = lbt_dec(quantise(Y_lbt, best_s), C8, Pr_lbt)

# --- DWT (equal-MSE, n=3) at matched RMS ---
n_dwt = 3
ratios_dwt = dwt_impulse_ratios(X.shape[0], n_dwt)
errs_dwt = []
for s in step_grid_fine:
    dwtstep = s * ratios_dwt
    dwtstep[1, n_dwt] = dwtstep[0, n_dwt]; dwtstep[2, n_dwt] = dwtstep[0, n_dwt]
    _, rms_s, _ = dwt_quantize_layered(X, dwtstep, n_dwt)
    errs_dwt.append(rms_s)
best_s = step_grid_fine[np.argmin(np.abs(np.array(errs_dwt) - rms_ref))]
dwtstep = best_s * ratios_dwt
dwtstep[1, n_dwt] = dwtstep[0, n_dwt]; dwtstep[2, n_dwt] = dwtstep[0, n_dwt]
Z_dwt, _, _ = dwt_quantize_layered(X, dwtstep, n_dwt)

# --- Plot zoomed crop (1 row x 6 cols) ---
methods = [
    ('Original', X),
    ('Direct', quantise(X, 17)),
    ('Pyramid', Z_pyr),
    ('DCT 8×8', Z_dct),
    ('LBT 8×8', Z_lbt),
    (f'DWT n={n_dwt}', Z_dwt),
]
fig, axs = plt.subplots(1, 6, figsize=(20, 3.5))
for ax, (label, Z) in zip(axs, methods):
    plot_image(Z[r0:r1, c0:c1], ax=ax)
    ax.set(title=label, xticks=[], yticks=[])
fig.suptitle(f'Zoomed crop comparison at matched RMS ≈ {rms_ref:.2f}', fontsize=12, y=1.02)
fig.tight_layout()
fig.savefig('zach/images/cross_method_crop.png', dpi=200, bbox_inches='tight')

### Quantifying block artefacts

Measure the mean absolute horizontal gradient $|Z_{i,j+1} - Z_{i,j}|$ at 8-pixel block boundaries vs non-boundary positions. A ratio > 1 indicates visible blocking; the LBT's post-filter should suppress this.


In [ ]:
def block_discontinuity(Z, N=8):
    """Ratio of mean absolute gradient at N-pixel boundaries vs non-boundaries."""
    grad = np.abs(np.diff(Z, axis=1))  # horizontal gradient, shape (m, n-1)
    cols = np.arange(grad.shape[1])
    boundary = (cols % N) == (N - 1)   # columns just before each block boundary
    mean_boundary = np.mean(grad[:, boundary])
    mean_interior = np.mean(grad[:, ~boundary])
    return mean_boundary / mean_interior

print(f'{"Method":<12} {"Boundary/Interior ratio":>24}')
print('-' * 38)
for label, Z in [('Original', X), ('Direct', quantise(X, 17)),
                  ('DCT 8×8', Z_dct), ('LBT 8×8', Z_lbt), ('DWT n=3', Z_dwt)]:
    r = block_discontinuity(Z)
    print(f'{label:<12} {r:>24.3f}')